# P1 — Gobierno de datos para Big Data

**Big Data — 7BM1 — 26-27/1**  
**Sesión 7 — 08/09/2026 — 90 min**

### Equipo
- Integrante 1: Adair Hernandez Valdivia
- Integrante 2: Kitzia Maria Araujo Perez
- Integrante 3: Luis Axel Zarate Lozano

## Pregunta central

> **¿Estos datos son suficientemente confiables y gobernables para utilizarlos en la aplicación propuesta de estimación de demanda y apoyo a decisiones operativas?**

El código es un instrumento de observación. La práctica evalúa principalmente la calidad de la evidencia, su interpretación, la relación problema → riesgo → regla de gobierno y la decisión final.


## 0. Archivos y reglas del caso

Archivos esperados en la misma carpeta del notebook:

- `P1_viajes.csv`
- `P1_estaciones.csv`

Reglas conocidas:

- unidades válidas: `U01`–`U12`;
- estaciones válidas: las presentes en `P1_estaciones.csv`;
- capacidad máxima: **80 pasajeros**;
- servicios previstos: `REGULAR` y `EXPRES`;
- un registro debería recibirse en **≤ 10 min** desde su inicio;
- las coordenadas de origen deben ser coherentes con la estación registrada;
- `viaje_id` debe identificar inequívocamente un viaje.

**Importante:** `ocupacion` aparece en el extracto, pero su unidad/escala no está documentada.


In [2]:
import pandas as pd

pd.set_option('display.max_columns', None)

viajes = pd.read_csv('P1_viajes.csv')
estaciones = pd.read_csv('P1_estaciones.csv')

print('viajes:', viajes.shape)
print('estaciones:', estaciones.shape)
display(viajes.head())
display(estaciones)

viajes: (60, 13)
estaciones: (8, 6)


,viaje_id,fecha_hora_inicio,fecha_hora_recepcion,unidad_id,estacion_origen_id,estacion_destino_id,lat_origen,lon_origen,tipo_servicio,pasajeros,ocupacion,fuente_registro,conductor_id
0,V0001,2026-09-03 06:00:00,2026-09-03 06:02:00,U01,E01,E04,19.4800,-99.1200,REGULAR,19,31,VALIDADOR,C001
1,V0002,2026-09-03 06:12:00,2026-09-03 06:15:00,U02,E02,E05,19.4325,-99.1332,REGULAR,26,42,INTEGRACION,C002
2,V0003,2026-09-03 06:24:00,2026-09-03 06:28:00,U03,E03,E06,19.4100,-99.0700,REGULAR,33,53,GPS_APP,C003
3,V0004,2026-09-03 06:36:00,2026-09-03 06:41:00,U04,E04,E07,19.3500,-99.1500,EXPRES,40,64,VALIDADOR,C004
4,V0005,2026-09-03 06:48:00,2026-09-03 06:54:00,U05,E05,E08,19.4200,-99.2100,REGULAR,47,75,NaN,C005


,estacion_id,nombre,lat_ref,lon_ref,zona,activa
0,E01,Estación Norte,19.4800,-99.1200,Norte,True
1,E02,Estación Centro,19.4325,-99.1332,Centro,True
2,E03,Estación Oriente,19.4100,-99.0700,Oriente,True
3,E04,Estación Sur,19.3500,-99.1500,Sur,True
4,E05,Estación Poniente,19.4200,-99.2100,Poniente,True
5,E06,Estación Universidad,19.3320,-99.1870,Sur,True
6,E07,Estación Mercado,19.4450,-99.1050,Centro,True
7,E08,Estación Terminal,19.5000,-99.1600,Norte,True


## Fase 1 — Comprender el ecosistema de datos (≈10 min)

Complete brevemente:

| Elemento | Identificación del equipo |
|---|---|
| Fuente(s) | Registro interno de la caja negra del camion asi como registros de bitacora |
| Dataset(s) | P1_estaciones.csv y P1_viajes.csv |
| Forma de almacenamiento recibida | Dos CSV con tablas estructuradas |
| Uso propuesto | Predicción de retrasos |
| Quién necesita confiar en los datos | Encargados de lineas y usuarios del transporte |
| Metadatos mínimos que deberían conservarse | Sensores, GPS, timestamps |

**Pregunta:** ¿qué relación existe entre la calidad del dataset y el valor que la organización espera obtener?

**Respuesta del equipo:**  
Directamente proporcional esto se debe a que entre mejor sean los datos la calidad de las predicciones sera mayor y la estrategias que se implementen seran mas efectivas


## Fase 2 — Perfilado del dataset (≈20 min)

Obtenga evidencia sobre:

1. dimensiones y tipos;
2. faltantes;
3. duplicados;
4. valores únicos y frecuencias;
5. mínimos/máximos;
6. fechas y oportunidad;
7. compatibilidad con catálogos/reglas conocidas.

No limpie todo el dataset. Primero **observe y documente**.


In [4]:
# 2.1 Dimensiones y tipos
print(viajes.shape)
print(viajes.dtypes)

print(estaciones.shape)
print(estaciones.dtypes)

(60, 13)
viaje_id                    str
fecha_hora_inicio           str
fecha_hora_recepcion        str
unidad_id                   str
estacion_origen_id          str
estacion_destino_id         str
lat_origen              float64
lon_origen              float64
tipo_servicio               str
pasajeros                 int64
ocupacion                 int64
fuente_registro             str
conductor_id                str
dtype: object
(8, 6)
estacion_id        str
nombre             str
lat_ref        float64
lon_ref        float64
zona               str
activa            bool
dtype: object


In [6]:
# 2.2 Valores faltantes
print(viajes.isnull().sum())

print(estaciones.isnull().sum())

viaje_id                0
fecha_hora_inicio       3
fecha_hora_recepcion    0
unidad_id               0
estacion_origen_id      0
estacion_destino_id     0
lat_origen              0
lon_origen              0
tipo_servicio           0
pasajeros               0
ocupacion               0
fuente_registro         3
conductor_id            0
dtype: int64
estacion_id    0
nombre         0
lat_ref        0
lon_ref        0
zona           0
activa         0
dtype: int64


In [8]:
# Si alguna cadena vacía no fue interpretada como NaN, puede comprobarla así:
print((viajes == '').sum())
print((estaciones == '').sum())

viaje_id                0
fecha_hora_inicio       0
fecha_hora_recepcion    0
unidad_id               0
estacion_origen_id      0
estacion_destino_id     0
lat_origen              0
lon_origen              0
tipo_servicio           0
pasajeros               0
ocupacion               0
fuente_registro         0
conductor_id            0
dtype: int64
estacion_id    0
nombre         0
lat_ref        0
lon_ref        0
zona           0
activa         0
dtype: int64


In [14]:
# 2.3 Duplicados exactos


# Explore también si viaje_id identifica inequívocamente un viaje.
# TODO: complete una comprobación de IDs repetidos.

print(viajes['viaje_id'].duplicated().sum())

print(viajes['viaje_id'].value_counts().head(10))


print(estaciones['estacion_id'].duplicated().sum())

print(estaciones['estacion_id'].value_counts())


2
viaje_id
V0011    2
V0025    2
V0001    1
V0002    1
V0003    1
V0004    1
V0005    1
V0006    1
V0007    1
V0008    1
Name: count, dtype: int64
0
estacion_id
E01    1
E02    1
E03    1
E04    1
E05    1
E06    1
E07    1
E08    1
Name: count, dtype: int64


In [19]:
# 2.4 Frecuencias de variables categóricas
print("---"*10)
print(viajes['tipo_servicio'].value_counts())
print("---"*10)

print(viajes['fuente_registro'].value_counts())
print("---"*10)
print(viajes['estacion_origen_id'].value_counts().head(10))
print("---"*10)

print(viajes['estacion_destino_id'].value_counts().head(10))
print("---"*10)

print(viajes['unidad_id'].value_counts().head(10))
print("---"*10)


------------------------------
tipo_servicio
REGULAR    42
EXPRES     14
Regular     1
regular     1
Expres      1
EXPRESS     1
Name: count, dtype: int64
------------------------------
fuente_registro
VALIDADOR      19
INTEGRACION    19
GPS_APP        19
Name: count, dtype: int64
------------------------------
estacion_origen_id
E03    9
E01    8
E02    8
E04    7
E05    7
E07    7
E08    7
E06    6
E99    1
Name: count, dtype: int64
------------------------------
estacion_destino_id
E04    8
E05    8
E06    8
E07    7
E08    7
E01    7
E02    7
E03    7
E00    1
Name: count, dtype: int64
------------------------------
unidad_id
U11    6
U01    5
U02    5
U03    5
U04    5
U05    5
U06    5
U08    5
U10    5
U07    4
Name: count, dtype: int64
------------------------------


In [ ]:
# 2.5 Rangos numéricosribe()

countDatos = [
    viajes['pasajeros'].count(),
    viajes['ocupacion'].count(),
    viajes['lat_origen'].count(),
    viajes['lon_origen'].count()
]
meanDatos = [
    viajes['pasajeros'].mean(),
    viajes['ocupacion'].mean(),
    viajes['lat_origen'].mean(),
    viajes['lon_origen'].mean()
]
stdDatos = [
    viajes['pasajeros'].std(),
    viajes['ocupacion'].std(),
    viajes['lat_origen'].std(),
    viajes['lon_origen'].std()
]
minDatos = [
    viajes['pasajeros'].min(),
    viajes['ocupacion'].min(),
    viajes['lat_origen'].min(),
    viajes['lon_origen'].min()
]
cuartiles
maxDatos = [
    viajes['pasajeros'].max(),
    viajes['ocupacion'].max(),
    viajes['lat_origen'].max(),
    viajes['lon_origen'].max()
]

60


### Fechas

Los registros pueden contener representaciones inconsistentes. Procure transformar las fechas **sin detener el notebook ante un valor no interpretable**. Una opción es usar `errors='coerce'` y revisar qué registros se convierten en `NaT`.

> No asuma que un valor que no pudo convertirse está necesariamente “mal” sin revisar el dato original y el contexto.


In [ ]:
# 2.6 Fechas: complete o adapte este bloque.
# Sugerencia: pruebe primero la conversión y compare el resultado con la columna original.

try:
    inicio_parseado = pd.to_datetime(
        viajes['fecha_hora_inicio'], format='mixed', errors='coerce', dayfirst=True
    )
except TypeError:
    # Compatibilidad con versiones anteriores de pandas.
    inicio_parseado = viajes['fecha_hora_inicio'].apply(
        lambda x: pd.to_datetime(x, errors='coerce', dayfirst=True)
    )

recepcion_parseada = pd.to_datetime(viajes['fecha_hora_recepcion'], errors='coerce')

print('Inicio no interpretable / ausente:', inicio_parseado.isna().sum())

# TODO: calcule el retraso de recepción en minutos y localice registros > 10 min.


Inicio no interpretable / ausente: 4


In [ ]:
# 2.7 Reglas y catálogos
unidades_validas = {f'U{i:02d}' for i in range(1, 13)}
estaciones_validas = set(estaciones['estacion_id'])
servicios_validos = {'REGULAR', 'EXPRES'}

# TODO: utilice estas referencias para localizar valores que no cumplen las reglas.
# Ejemplo de patrón (adáptelo a la columna que necesite):
# viajes[~viajes['...'].isin(...)]


In [ ]:
# 2.8 Comparación con el catálogo maestro de estaciones
# Puede combinar viajes con estaciones usando estacion_origen_id ↔ estacion_id.
# Después compare lat_origen/lon_origen con lat_ref/lon_ref.

# TODO: construya la comparación y conserve evidencia de los casos que le parezcan relevantes.


## Fase 3 — Auditoría de calidad (≈20 min)

Registre **al menos seis hallazgos no redundantes**. No gana puntos por acumular filas sin interpretación.

Dimensiones trabajadas: **completitud, validez, consistencia, oportunidad y exactitud**.

- Use **exactitud** sólo si existe una referencia que permita contrastar el dato.
- Si la evidencia no permite determinar algo, escriba **“no puede determinarse con la evidencia disponible”**.

Complete la lista y ejecute la celda para visualizar su matriz.


In [ ]:
hallazgos = [
    # Ejemplo de estructura (NO corresponde a una respuesta del dataset):
    # {
    #     'Hallazgo': '...',
    #     'Dimension_criterio': '...',
    #     'Evidencia': '...',
    #     'Impacto': '...'
    # },
]

matriz_hallazgos = pd.DataFrame(hallazgos)
matriz_hallazgos

""


## Fase 4 — Gobierno, seguridad y ética (≈20 min)

Para esta práctica, **gobierno de datos** significa establecer responsabilidades, metadatos y reglas para decidir cómo se reciben, validan, utilizan y protegen los datos.

Proponga **al menos cuatro reglas/controles**, incluyendo:

1. una de validación;
2. una de metadatos/procedencia;
3. una de acceso o protección;
4. una derivada directamente de otro hallazgo.

Analice también:

- quién debería ser responsable de cada regla;
- en qué momento debería aplicarse;
- qué atributos necesitan realmente los usuarios de la aplicación;
- si algún atributo debería eliminarse, restringirse o protegerse por no ser necesario para el propósito.


In [ ]:
reglas_gobierno = [
    # {
    #     'Problema_riesgo': '...',
    #     'Regla_control': '...',
    #     'Responsable': '...',
    #     'Momento_aplicacion': '...'
    # },
]

matriz_gobierno = pd.DataFrame(reglas_gobierno)
matriz_gobierno

### Preguntas de análisis de acceso y uso responsable

1. ¿`conductor_id` es necesario para estimar demanda y apoyar la decisión operativa planteada? ¿Qué harían con esa variable y por qué?

**Respuesta:**  

2. ¿La ausencia de `fuente_registro` afecta sólo la calidad o también la gobernanza del dato? Explique.

**Respuesta:**  

3. ¿Qué información adicional necesitarían antes de afirmar que `ocupacion` está bien o mal registrada?

**Respuesta:**  



## Fase 5 — Decisión final (≈15 min)

Seleccione una opción:

- [ ] **Sí**
- [ ] **Sí, con condiciones**
- [ ] **No todavía**

### Conclusión (aprox. 150–200 palabras)

Incluya:

- al menos **tres evidencias** obtenidas durante la auditoría;
- problemas principales;
- riesgos para la aplicación propuesta;
- controles imprescindibles;
- alcance de la autorización.

**Conclusión del equipo:**  



## Reflexión final — sin puntaje independiente

> **¿Que un dataset sea públicamente accesible significa automáticamente que sea confiable, de calidad y apropiado para cualquier uso?**

**Respuesta breve:**  



## Checklist antes de entregar

- [ ] El equipo está identificado.
- [ ] Las celdas relevantes conservan sus salidas.
- [ ] Hay al menos 6 hallazgos no redundantes con evidencia e impacto.
- [ ] Hay al menos 4 reglas/controles de gobierno.
- [ ] La decisión final usa evidencia del propio notebook.
- [ ] No se atribuyó “exactitud” sin una referencia suficiente.
- [ ] El archivo se guardó como `P1_EquipoXX_GobiernoDatos.ipynb`.
- [ ] Un integrante entregará el notebook en Google Classroom según la hora límite indicada por el docente.
